# Coronary Artery Disease and Myocardial Infarction Associations from CARDIoGRAMplusC4D GWAS

This notebook extracts coronary artery disease (CAD) and myocardial infarction (MI) GWAS associations from the CARDIoGRAMplusC4D summary statistics files for all gene regions identified in the Ensembl gene-region step.

In [1]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", None)

In [2]:
gene_regions_file = Path("../data/interim/ensembl/gene_regions_grch38.csv")

cad_path = Path("../data/raw/cardiogram_c4d/cardiogram_cad_harmonized.tsv")
mi_path = Path("../data/raw/cardiogram_c4d/cardiogram_mi_harmonized.tsv")

out_dir = Path("../data/interim/cardiogram_c4d")
out_dir.mkdir(parents=True, exist_ok=True)

out_cad_csv = out_dir / "cad_associations.csv"
out_mi_csv = out_dir / "mi_associations.csv"

In [3]:
gene_regions = pd.read_csv(gene_regions_file)

gene_regions.head()

,input_gene_symbol,official_gene_symbol,ensembl_gene_id,lookup_status,species,chromosome,gene_start,gene_end,strand,strand_symbol,assembly_name,flanking_bp,region_start,region_end,upstream_start,upstream_end,downstream_start,downstream_end,biotype,object_type,source
0,APOE,APOE,ENSG00000130203,found,homo_sapiens,19,44903787,44909396,1,+,GRCh38,50000,44853787,44959396,44853787,44903786,44909397,44959396,protein_coding,Gene,Ensembl REST
1,B2M,B2M,ENSG00000166710,found,homo_sapiens,15,44711253,44718851,1,+,GRCh38,50000,44661253,44768851,44661253,44711252,44718852,44768851,protein_coding,Gene,Ensembl REST
2,C1QA,C1QA,ENSG00000173372,found,homo_sapiens,1,22635057,22639681,1,+,GRCh38,50000,22585057,22689681,22585057,22635056,22639682,22689681,protein_coding,Gene,Ensembl REST
3,C1QB,C1QB,ENSG00000173369,found,homo_sapiens,1,22652713,22661637,1,+,GRCh38,50000,22602713,22711637,22602713,22652712,22661638,22711637,protein_coding,Gene,Ensembl REST
4,C1QC,C1QC,ENSG00000159189,found,homo_sapiens,1,22642558,22648110,1,+,GRCh38,50000,22592558,22698110,22592558,22642557,22648111,22698110,protein_coding,Gene,Ensembl REST


In [4]:
genes_for_cardiogram = gene_regions.loc[
    gene_regions["lookup_status"] == "found"
].copy()

genes_for_cardiogram = (
    genes_for_cardiogram
    .dropna(subset=["official_gene_symbol", "chromosome", "region_start", "region_end"])
    .drop_duplicates(subset=["official_gene_symbol"])
    .reset_index(drop=True)
)

print("Genes available for CARDIoGRAMplusC4D filtering:", len(genes_for_cardiogram))

genes_for_cardiogram.head()

Genes available for CARDIoGRAMplusC4D filtering: 52


,input_gene_symbol,official_gene_symbol,ensembl_gene_id,lookup_status,species,chromosome,gene_start,gene_end,strand,strand_symbol,assembly_name,flanking_bp,region_start,region_end,upstream_start,upstream_end,downstream_start,downstream_end,biotype,object_type,source
0,APOE,APOE,ENSG00000130203,found,homo_sapiens,19,44903787,44909396,1,+,GRCh38,50000,44853787,44959396,44853787,44903786,44909397,44959396,protein_coding,Gene,Ensembl REST
1,B2M,B2M,ENSG00000166710,found,homo_sapiens,15,44711253,44718851,1,+,GRCh38,50000,44661253,44768851,44661253,44711252,44718852,44768851,protein_coding,Gene,Ensembl REST
2,C1QA,C1QA,ENSG00000173372,found,homo_sapiens,1,22635057,22639681,1,+,GRCh38,50000,22585057,22689681,22585057,22635056,22639682,22689681,protein_coding,Gene,Ensembl REST
3,C1QB,C1QB,ENSG00000173369,found,homo_sapiens,1,22652713,22661637,1,+,GRCh38,50000,22602713,22711637,22602713,22652712,22661638,22711637,protein_coding,Gene,Ensembl REST
4,C1QC,C1QC,ENSG00000159189,found,homo_sapiens,1,22642558,22648110,1,+,GRCh38,50000,22592558,22698110,22592558,22642557,22648111,22698110,protein_coding,Gene,Ensembl REST


In [5]:
for path in [cad_path, mi_path]:
    if not path.exists():
        raise FileNotFoundError(f"Input file not found: {path}")

In [6]:
cardiogram_columns = [
    "hm_variant_id",
    "hm_rsid",
    "hm_chrom",
    "hm_pos",
    "hm_other_allele",
    "hm_effect_allele",
    "hm_beta",
    "hm_odds_ratio",
    "hm_ci_lower",
    "hm_ci_upper",
    "hm_effect_allele_frequency",
    "hm_code",
    "standard_error",
    "p_value",
]

In [7]:
def normalize_chromosome(value):
    value = str(value).strip()
    value = value.replace("chr", "")

    if value.endswith(".0"):
        value = value[:-2]

    return value

In [8]:
def annotate_location_relative_to_gene(position, gene_start, gene_end, strand):
    """
    Annotate whether a variant is inside the gene body,
    upstream, or downstream, taking gene strand into account.
    """
    if pd.isna(position) or pd.isna(gene_start) or pd.isna(gene_end):
        return None

    position = int(position)
    gene_start = int(gene_start)
    gene_end = int(gene_end)

    if gene_start <= position <= gene_end:
        return "intragenic"

    if str(strand) in ["1", "+", "+1"]:
        if position < gene_start:
            return "upstream"
        return "downstream"

    if str(strand) in ["-1", "-", "−"]:
        if position > gene_end:
            return "upstream"
        return "downstream"

    return None


def calculate_distance_from_gene(position, gene_start, gene_end):
    """
    Calculate distance from the gene body.
    Variants inside the gene body have distance 0.
    """
    if pd.isna(position) or pd.isna(gene_start) or pd.isna(gene_end):
        return None

    position = int(position)
    gene_start = int(gene_start)
    gene_end = int(gene_end)

    if gene_start <= position <= gene_end:
        return 0

    if position < gene_start:
        return gene_start - position

    return position - gene_end

In [9]:
regions_for_filtering = genes_for_cardiogram.copy()

regions_for_filtering["chromosome_filter"] = (
    regions_for_filtering["chromosome"]
    .astype(str)
    .str.replace("chr", "", regex=False)
)

regions_for_filtering["region_start"] = pd.to_numeric(
    regions_for_filtering["region_start"],
    errors="coerce"
)

regions_for_filtering["region_end"] = pd.to_numeric(
    regions_for_filtering["region_end"],
    errors="coerce"
)

regions_for_filtering.head()

,input_gene_symbol,official_gene_symbol,ensembl_gene_id,lookup_status,species,chromosome,gene_start,gene_end,strand,strand_symbol,assembly_name,flanking_bp,region_start,region_end,upstream_start,upstream_end,downstream_start,downstream_end,biotype,object_type,source,chromosome_filter
0,APOE,APOE,ENSG00000130203,found,homo_sapiens,19,44903787,44909396,1,+,GRCh38,50000,44853787,44959396,44853787,44903786,44909397,44959396,protein_coding,Gene,Ensembl REST,19
1,B2M,B2M,ENSG00000166710,found,homo_sapiens,15,44711253,44718851,1,+,GRCh38,50000,44661253,44768851,44661253,44711252,44718852,44768851,protein_coding,Gene,Ensembl REST,15
2,C1QA,C1QA,ENSG00000173372,found,homo_sapiens,1,22635057,22639681,1,+,GRCh38,50000,22585057,22689681,22585057,22635056,22639682,22689681,protein_coding,Gene,Ensembl REST,1
3,C1QB,C1QB,ENSG00000173369,found,homo_sapiens,1,22652713,22661637,1,+,GRCh38,50000,22602713,22711637,22602713,22652712,22661638,22711637,protein_coding,Gene,Ensembl REST,1
4,C1QC,C1QC,ENSG00000159189,found,homo_sapiens,1,22642558,22648110,1,+,GRCh38,50000,22592558,22698110,22592558,22642557,22648111,22698110,protein_coding,Gene,Ensembl REST,1


In [10]:
def process_cardiogram_dataset(input_path, column_prefix):
    matched_chunks = []

    metadata_columns = [
        "query_source",
        "query_value",
        "region_assembly",
        "gene_start",
        "gene_end",
        "region_start",
        "region_end",
        "strand",
        "strand_symbol",
        "location_relative_to_gene",
        "distance_from_gene",
    ]

    for chunk in pd.read_csv(
        input_path,
        sep="\t",
        usecols=lambda col: col in cardiogram_columns,
        na_values=["NA", "N/A", ""],
        chunksize=500_000,
        low_memory=False
    ):
        chunk["hm_chrom_filter"] = chunk["hm_chrom"].apply(normalize_chromosome)

        chunk["hm_pos_filter"] = pd.to_numeric(
            chunk["hm_pos"],
            errors="coerce"
        )

        for _, gene_region in regions_for_filtering.iterrows():
            region_chromosome = gene_region["chromosome_filter"]
            region_start = int(gene_region["region_start"])
            region_end = int(gene_region["region_end"])

            matched = chunk[
                (chunk["hm_chrom_filter"] == region_chromosome)
                & (chunk["hm_pos_filter"].between(region_start, region_end))
            ].copy()

            if not matched.empty:
                matched["query_source"] = "gene_region"
                matched["query_value"] = gene_region["official_gene_symbol"]
                matched["region_assembly"] = gene_region["assembly_name"]
                matched["gene_start"] = int(gene_region["gene_start"])
                matched["gene_end"] = int(gene_region["gene_end"])
                matched["region_start"] = region_start
                matched["region_end"] = region_end
                matched["strand"] = gene_region["strand"]
                matched["strand_symbol"] = gene_region["strand_symbol"]

                matched["location_relative_to_gene"] = matched["hm_pos_filter"].apply(
                    lambda position: annotate_location_relative_to_gene(
                        position,
                        gene_region["gene_start"],
                        gene_region["gene_end"],
                        gene_region["strand"]
                    )
                )

                matched["distance_from_gene"] = matched["hm_pos_filter"].apply(
                    lambda position: calculate_distance_from_gene(
                        position,
                        gene_region["gene_start"],
                        gene_region["gene_end"]
                    )
                )

                matched = matched.drop(
                    columns=["hm_chrom_filter", "hm_pos_filter"]
                )

                matched_chunks.append(matched)

    if matched_chunks:
        associations_df = pd.concat(matched_chunks, ignore_index=True)
    else:
        associations_df = pd.DataFrame(columns=cardiogram_columns + metadata_columns)

    rename_map = {
        col: f"{column_prefix}_{col}"
        for col in cardiogram_columns
    }

    associations_df = associations_df.rename(columns=rename_map)

    numeric_cols = [
        f"{column_prefix}_hm_chrom",
        f"{column_prefix}_hm_pos",
        f"{column_prefix}_hm_beta",
        f"{column_prefix}_hm_odds_ratio",
        f"{column_prefix}_hm_ci_lower",
        f"{column_prefix}_hm_ci_upper",
        f"{column_prefix}_hm_effect_allele_frequency",
        f"{column_prefix}_standard_error",
        f"{column_prefix}_p_value",
        "gene_start",
        "gene_end",
        "region_start",
        "region_end",
        "distance_from_gene",
    ]

    for col in numeric_cols:
        if col in associations_df.columns:
            associations_df[col] = pd.to_numeric(
                associations_df[col],
                errors="coerce"
            )

    rows_before_deduplication = len(associations_df)

    associations_df = associations_df.drop_duplicates()

    rows_after_deduplication = len(associations_df)
    duplicate_rows_removed = rows_before_deduplication - rows_after_deduplication

    p_value_col = f"{column_prefix}_p_value"

    if p_value_col in associations_df.columns:
        associations_df = associations_df.sort_values(
            ["query_value", p_value_col],
            ascending=[True, True],
            na_position="last"
        ).reset_index(drop=True)

    return associations_df, duplicate_rows_removed

In [11]:
cad_associations_df, cad_duplicate_rows_removed = process_cardiogram_dataset(
    input_path=cad_path,
    column_prefix="CAD"
)

cad_associations_df.to_csv(out_cad_csv, index=False)

cad_associations_df.shape

(22273, 25)

In [12]:
mi_associations_df, mi_duplicate_rows_removed = process_cardiogram_dataset(
    input_path=mi_path,
    column_prefix="MI"
)

mi_associations_df.to_csv(out_mi_csv, index=False)

mi_associations_df.shape

(21034, 25)

In [13]:
cad_genes_with = set(cad_associations_df["query_value"].dropna().unique())
mi_genes_with = set(mi_associations_df["query_value"].dropna().unique())
all_genes = set(genes_for_cardiogram["official_gene_symbol"])

summary = {
    "region_assembly": "GRCh38",
    "genes_available_for_filtering": int(len(genes_for_cardiogram)),
    "cad": {
        "phenotype": "Coronary artery disease",
        "genes_with_cad_associations": int(cad_associations_df["query_value"].nunique()),
        "genes_without_cad_associations": sorted(all_genes - cad_genes_with),
        "associations": int(len(cad_associations_df)),
        "unique_rsIDs": int(cad_associations_df["CAD_hm_rsid"].nunique()),
        "duplicate_rows_removed": int(cad_duplicate_rows_removed),
    },
    "mi": {
        "phenotype": "Myocardial infarction",
        "genes_with_mi_associations": int(mi_associations_df["query_value"].nunique()),
        "genes_without_mi_associations": sorted(all_genes - mi_genes_with),
        "associations": int(len(mi_associations_df)),
        "unique_rsIDs": int(mi_associations_df["MI_hm_rsid"].nunique()),
        "duplicate_rows_removed": int(mi_duplicate_rows_removed),
    },
}

summary

{'region_assembly': 'GRCh38',
 'genes_available_for_filtering': 52,
 'cad': {'phenotype': 'Coronary artery disease',
  'genes_with_cad_associations': 51,
  'associations': 22273,
  'unique_rsIDs': 21196,
  'duplicate_rows_removed': 0},
 'mi': {'phenotype': 'Myocardial infarction',
  'genes_with_mi_associations': 51,
  'associations': 21034,
  'unique_rsIDs': 20010,
  'duplicate_rows_removed': 0}}